In [0]:
%pip install mlflow openai --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import json
import pandas as pd
from openai import OpenAI

client = OpenAI(
    base_url="https://dbc-820598a7-e451.cloud.databricks.com/serving-endpoints",
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

# Load data
df = spark.table("default.ghana_gap_analysis").toPandas()
print("Loaded:", len(df), "facilities")

mlflow.set_experiment("/IDP_Agent_Citations")

CITATION_PROMPT = """You are a medical facility information extractor.
Extract structured facts from the facility information below.

For EACH fact you extract, you MUST cite the exact source field it came from.

Return ONLY a valid JSON object in this exact format:
{
  "procedure": [
    {"fact": "the procedure you extracted", "citation": "exact text from source that supports this"}
  ],
  "equipment": [
    {"fact": "the equipment you extracted", "citation": "exact text from source that supports this"}
  ],
  "capability": [
    {"fact": "the capability you extracted", "citation": "exact text from source that supports this"}
  ],
  "reasoning_steps": [
    "Step 1: I analyzed the facility name and type...",
    "Step 2: I found procedures mentioned in...",
    "Step 3: I identified equipment from...",
    "Step 4: I determined capabilities based on..."
  ]
}

Rules:
- Every fact MUST have a citation
- Citations must be direct quotes from the source text
- reasoning_steps must show your thinking process
- Return empty lists if nothing found
"""

def run_idp_with_citations(row):
    facility_name = row.get('name', 'Unknown')
    
    # Build source context - each field is labeled so LLM can cite it
    source_text = f"""
[FIELD: facility_name] {row.get('name', '')}
[FIELD: facility_type] {row.get('facilityTypeId', '')}
[FIELD: city] {row.get('address_city', '')}
[FIELD: region] {row.get('address_stateOrRegion', '')}
[FIELD: description] {row.get('description', '')}
[FIELD: existing_procedures] {row.get('procedure', '')}
[FIELD: existing_equipment] {row.get('equipment', '')}
[FIELD: existing_capabilities] {row.get('capability', '')}
[FIELD: specialties] {row.get('specialties', '')}
[FIELD: idp_procedures] {row.get('idp_procedures', '')}
[FIELD: idp_equipment] {row.get('idp_equipment', '')}
[FIELD: idp_capabilities] {row.get('idp_capabilities', '')}
"""

    with mlflow.start_run(run_name=f"citation_{facility_name[:30]}", nested=True):
        # Log inputs
        mlflow.log_param("facility_name", facility_name)
        mlflow.log_param("city", row.get('address_city', ''))
        mlflow.log_text(source_text, "input_source.txt")

        # Step 1 - Extract with citations
        with mlflow.start_run(run_name="step1_extraction", nested=True):
            mlflow.log_text("Extracting facts with citations from source fields", "step_description.txt")
            
            response = client.chat.completions.create(
                model="databricks-meta-llama-3-3-70b-instruct",
                messages=[
                    {"role": "system", "content": CITATION_PROMPT},
                    {"role": "user", "content": f"Extract facts with citations for: {facility_name}\n\nSource data:\n{source_text}"}
                ],
                max_tokens=800
            )
            raw = response.choices[0].message.content.strip()
            mlflow.log_text(raw, "llm_raw_response.txt")

        # Step 2 - Parse response
        with mlflow.start_run(run_name="step2_parsing", nested=True):
            mlflow.log_text("Parsing and validating JSON response", "step_description.txt")
            try:
                clean = raw.replace("```json", "").replace("```", "").strip()
                parsed = json.loads(clean)
            except:
                parsed = {"procedure": [], "equipment": [], "capability": [], "reasoning_steps": []}
            mlflow.log_dict(parsed, "parsed_output.json")

        # Step 3 - Build citation report
        with mlflow.start_run(run_name="step3_citation_report", nested=True):
            mlflow.log_text("Building human-readable citation report", "step_description.txt")
            
            report = f"CITATION REPORT: {facility_name}\n"
            report += "=" * 60 + "\n\n"
            
            report += "REASONING STEPS:\n"
            for i, step in enumerate(parsed.get('reasoning_steps', []), 1):
                report += f"  {step}\n"
            
            report += "\nEXTRACTED FACTS WITH CITATIONS:\n"
            for category in ['procedure', 'equipment', 'capability']:
                facts = parsed.get(category, [])
                if facts:
                    report += f"\n{category.upper()}:\n"
                    for item in facts:
                        if isinstance(item, dict):
                            report += f"  ✓ FACT: {item.get('fact', '')}\n"
                            report += f"    📎 CITATION: \"{item.get('citation', '')}\"\n"
            
            mlflow.log_text(report, "citation_report.txt")

        return parsed, report

# Test on 3 facilities
print("Testing citation agent on 3 facilities...\n")
with mlflow.start_run(run_name="Citation_Agent_Test"):
    for _, row in df.head(3).iterrows():
        result, report = run_idp_with_citations(row)
        print(report)
        print()

Loaded: 1067 facilities


2026/03/11 14:51:16 INFO mlflow.tracking.fluent: Experiment with name '/IDP_Agent_Citations' does not exist. Creating a new experiment.


Testing citation agent on 3 facilities...

CITATION REPORT: 109/No 1 Bekwai Rd (Near Mexico Hotel) Takoradi, Ghana

REASONING STEPS:
  Step 1: I analyzed the facility name and type from the facility_name and facility_type fields to understand the context.
  Step 2: I found procedures mentioned in the idp_procedures field, which included HIV Testing and Counseling, Behavior Change Communication, Community Outreach, and Hospice / Home Based Care.
  Step 3: I identified equipment from the idp_equipment field, but it was empty.
  Step 4: I determined capabilities based on the idp_capabilities field, which included care and support services, prevention services, stigma reduction services, public health services, and global health services.

EXTRACTED FACTS WITH CITATIONS:

PROCEDURE:
  ✓ FACT: HIV Testing and Counseling
    📎 CITATION: "HIV Testing and Counseling"
  ✓ FACT: Behavior Change Communication
    📎 CITATION: "Behavior Change Communication"
  ✓ FACT: Community Outreach
    📎 CITAT

In [0]:
import time

print("Running Citation Agent on all facilities...")
all_results = []

with mlflow.start_run(run_name="Citation_Agent_Full_Run"):
    for i, (_, row) in enumerate(df.iterrows()):
        try:
            result, report = run_idp_with_citations(row)
            
            # Flatten cited facts
            procedures = [item.get('fact','') for item in result.get('procedure',[]) if isinstance(item, dict)]
            equipment = [item.get('fact','') for item in result.get('equipment',[]) if isinstance(item, dict)]
            capabilities = [item.get('fact','') for item in result.get('capability',[]) if isinstance(item, dict)]
            proc_citations = [item.get('citation','') for item in result.get('procedure',[]) if isinstance(item, dict)]
            equip_citations = [item.get('citation','') for item in result.get('equipment',[]) if isinstance(item, dict)]
            cap_citations = [item.get('citation','') for item in result.get('capability',[]) if isinstance(item, dict)]
            reasoning = result.get('reasoning_steps', [])

            all_results.append({
                "unique_id": row.get('unique_id', ''),
                "name": row.get('name', ''),
                "address_city": row.get('address_city', ''),
                "address_stateOrRegion": row.get('address_stateOrRegion', ''),
                "cited_procedures": json.dumps(procedures),
                "cited_equipment": json.dumps(equipment),
                "cited_capabilities": json.dumps(capabilities),
                "procedure_citations": json.dumps(proc_citations),
                "equipment_citations": json.dumps(equip_citations),
                "capability_citations": json.dumps(cap_citations),
                "reasoning_steps": json.dumps(reasoning),
                "citation_report": report,
            })
        except Exception as e:
            all_results.append({
                "unique_id": row.get('unique_id', ''),
                "name": row.get('name', ''),
                "address_city": row.get('address_city', ''),
                "address_stateOrRegion": row.get('address_stateOrRegion', ''),
                "cited_procedures": "[]", "cited_equipment": "[]",
                "cited_capabilities": "[]", "procedure_citations": "[]",
                "equipment_citations": "[]", "capability_citations": "[]",
                "reasoning_steps": "[]", "citation_report": f"Error: {e}",
            })

        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/1067 facilities...")
        
        time.sleep(0.3)

# Save
results_df = pd.DataFrame(all_results)
spark.createDataFrame(results_df).write.mode("overwrite").saveAsTable("default.ghana_citations")
print(f"\nDone! Saved {len(results_df)} citation reports")
print("Saved to: default.ghana_citations")

Running Citation Agent on all facilities...
  Processed 50/1067 facilities...
  Processed 100/1067 facilities...
  Processed 150/1067 facilities...
  Processed 200/1067 facilities...


  Processed 250/1067 facilities...
  Processed 300/1067 facilities...
  Processed 350/1067 facilities...
  Processed 400/1067 facilities...
  Processed 450/1067 facilities...
  Processed 500/1067 facilities...
  Processed 550/1067 facilities...
  Processed 600/1067 facilities...
  Processed 650/1067 facilities...
  Processed 700/1067 facilities...
  Processed 750/1067 facilities...
  Processed 800/1067 facilities...
  Processed 850/1067 facilities...
  Processed 900/1067 facilities...
  Processed 950/1067 facilities...
  Processed 1000/1067 facilities...
  Processed 1050/1067 facilities...

Done! Saved 1067 citation reports
Saved to: default.ghana_citations
